# 00 — Colab bootstrap (Layer 1: environment only)

Interactive, step-by-step equivalent of the required `colab_setup.py` bootstrap
(Master Engineering Plan `DOC-ARCH-000` v4.2.0: **§4.3**, **§8.5.1**, **ADR-0012**, **Gate P0**).

**What this notebook does**

1. Confirms it is running in Colab and reports the Python version.
2. Mounts Google Drive (idempotent).
3. Ensures `uv` is available.
4. Gets the repo onto the **ephemeral session disk** and runs `uv sync` from the committed `pyproject.toml` / `uv.lock`.
5. Resolves and creates the three **persistent Drive roots** (`FRP_DATA_ROOT`, `FRP_MODELS_ROOT`, `FRP_EVIDENCE_ROOT`).
6. Runs the Gate P0 write / read-back / SHA-256 smoke test on those roots.

**What it deliberately does not do:** download datasets, create `bronze/silver/gold` folders, run pipelines, or touch secrets. Those belong to later layers.

**Storage model (ADR-0012):** the Git clone, `.venv`, `uv` cache and scratch files live on the session disk and are recreated every session. Only the three roots below live on Drive and survive a reset.

Run the cells top to bottom (**Runtime → Run all** is fine). Every cell is safe to re-run.

## 1 · Environment check

Verifies the `google.colab` module is present and prints the runtime Python version. Per §8.5.1 the bootstrap exits with an actionable message outside a supported Colab runtime.

Shared settings are defined here. They can be overridden with environment variables, which is also how the notebook is dry-run outside Colab:
`FRP_ALLOW_NON_COLAB=1`, `FRP_SESSION_DIR`, `FRP_DRIVE_BASE`, `FRP_REPO_DIR`.

In [ ]:
import importlib.util
import os
import platform
import sys
from pathlib import Path


def _detect_colab() -> bool:
    if "google.colab" in sys.modules:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ImportError, ValueError):
        return False


IN_COLAB = _detect_colab()
ALLOW_NON_COLAB = os.environ.get("FRP_ALLOW_NON_COLAB") == "1"  # dry-run/testing only

# Ephemeral session disk: git clone, .venv, uv cache, scratch.
SESSION_DIR = Path(os.environ.get("FRP_SESSION_DIR", "/content"))

print(f"Python    : {sys.version.split()[0]}  ({sys.executable})")
print(f"Platform  : {platform.platform()}")
print(f"In Colab  : {IN_COLAB}")
print(f"Session disk (ephemeral): {SESSION_DIR}")

if not IN_COLAB and not ALLOW_NON_COLAB:
    raise RuntimeError(
        "This notebook must run in a Google Colab runtime (google.colab not importable). "
        "Open it in Colab (or the VS Code Colab extension connected to a Colab runtime) and re-run."
    )
if not IN_COLAB:
    print("WARNING: non-Colab dry-run mode (FRP_ALLOW_NON_COLAB=1). Drive mounting is skipped.")

## 2 · Mount Google Drive

Mounts Drive at `/content/drive`. If it is already mounted the cell does nothing, so re-running never fails or re-prompts. Drive is a FUSE/API-backed filesystem: it is **not** treated as native POSIX (no atomic-rename, locking or `fsync` guarantees — ADR-0012).

In [ ]:
DRIVE_MOUNT = "/content/drive"
MYDRIVE = f"{DRIVE_MOUNT}/MyDrive"


def drive_ready() -> bool:
    return os.path.ismount(DRIVE_MOUNT) and os.path.isdir(MYDRIVE)


if not IN_COLAB:
    print("Dry-run: skipping Drive mount.")
elif drive_ready():
    print(f"Drive already mounted at {DRIVE_MOUNT} - nothing to do.")
else:
    from google.colab import drive

    drive.mount(DRIVE_MOUNT)
    if not drive_ready():
        raise RuntimeError(
            f"Drive mount did not produce {MYDRIVE}. Re-run this cell and approve the "
            "authorization prompt."
        )
    print(f"Drive mounted at {DRIVE_MOUNT}.")

## 3 · Install `uv` and verify it

Colab does not ship `uv`. It is installed with pip only if missing, then invoked as `python -m uv` (or the `uv` binary if already on `PATH`), which avoids `PATH` surprises inside notebook subprocesses.

`run()` streams a command's output live and raises on a non-zero exit code.

In [ ]:
import shutil
import subprocess


def run(cmd, cwd=None, env=None, check=True) -> str:
    """Run a command, stream its output, return it; raise on failure when check=True."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {' '.join(cmd)}")
    return "".join(lines)


def uv_cmd():
    exe = shutil.which("uv")
    return [exe] if exe else [sys.executable, "-m", "uv"]


probe = subprocess.run(uv_cmd() + ["--version"], capture_output=True, text=True)
if probe.returncode != 0:
    print("uv not found - installing with pip ...")
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

UV = uv_cmd()
run(UV + ["--version"])

## 4 · Repository and dependencies (`uv sync`)

- If the notebook is already inside a checkout (a parent folder has `pyproject.toml`), that checkout is used. Otherwise the repo is cloned to the **session disk** (`/content/fraud-risk-platform`). An existing clone is only fast-forwarded, never overwritten.
- Dependencies come from the committed `pyproject.toml` and `uv.lock` via `uv sync --frozen`. If `uv.lock` is missing the cell falls back to `uv sync` and warns that the result is **not reproducible** (§8.5.1 requires the lockfile).
- `.venv` stays on the session disk. If the project happens to sit on Drive, `UV_PROJECT_ENVIRONMENT` redirects the environment off Drive, and the `uv` cache is also kept on the session disk.
- If `.python-version` pins a Python that differs from Colab's, `uv` may download its own interpreter. That is expected and is why the interpreter printed in cell 1 is not necessarily the one in `.venv`.
- The clone assumes a public repo. For a private repo authenticate first; do not paste tokens into notebook cells.

In [ ]:
REPO_URL = os.environ.get("FRP_REPO_URL", "https://github.com/ebramrafat653-wq/fraud-risk-platform.git")
DEFAULT_CLONE_DIR = SESSION_DIR / "fraud-risk-platform"


def find_project_root() -> Path:
    override = os.environ.get("FRP_REPO_DIR")
    if override:
        return Path(override).resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").is_file():
            return p
    return DEFAULT_CLONE_DIR


PROJECT_ROOT = find_project_root()

if (PROJECT_ROOT / "pyproject.toml").is_file():
    print(f"Using existing checkout: {PROJECT_ROOT}")
    if (PROJECT_ROOT / ".git").exists():
        run(["git", "-C", PROJECT_ROOT, "status", "-sb"], check=False)
        run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"], check=False)  # never overwrites
else:
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"{PROJECT_ROOT} exists but has no pyproject.toml; refusing to overwrite it.")
    SESSION_DIR.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", REPO_URL, PROJECT_ROOT])

# --- uv environment: everything on the session disk -------------------------------------------
on_drive = str(PROJECT_ROOT).startswith(DRIVE_MOUNT + "/")
uv_env = os.environ.copy()
uv_env["UV_LINK_MODE"] = "copy"  # avoid cross-filesystem hardlink warnings
uv_env["UV_CACHE_DIR"] = str(SESSION_DIR / ".uv-cache")
if on_drive:
    uv_env["UV_PROJECT_ENVIRONMENT"] = str(SESSION_DIR / ".venvs" / "fraud-risk-platform")
    print("Project is on Drive -> redirecting the virtualenv to the session disk.")

lockfile = PROJECT_ROOT / "uv.lock"
if lockfile.is_file():
    sync_cmd = UV + ["sync", "--frozen"]
else:
    print("WARNING: uv.lock not found. Falling back to an unlocked `uv sync` - NOT reproducible. "
          "Commit uv.lock (Plan section 8.5.1).")
    sync_cmd = UV + ["sync"]

run(sync_cmd, cwd=PROJECT_ROOT, env=uv_env)

venv_prefix = run(
    UV + ["run", "--no-sync", "python", "-c", "import sys; print(sys.prefix)"],
    cwd=PROJECT_ROOT, env=uv_env,
).strip().splitlines()[-1]
if venv_prefix.startswith(DRIVE_MOUNT + "/"):
    raise RuntimeError(f"Virtualenv ended up on Drive ({venv_prefix}); it must live on the session disk.")
print(f"\nVirtualenv (session disk): {venv_prefix}")

## 5 · Persistent roots (`FRP_*_ROOT`)

Resolves the three logical roots to **Drive-backed** folders and creates them with `mkdir -p`. Defaults live under `MyDrive/fraud-risk-platform/`; pre-set environment variables win, so a different layout needs no code change.

| Variable | Default |
|---|---|
| `FRP_DATA_ROOT` | `…/fraud-risk-platform/data` |
| `FRP_MODELS_ROOT` | `…/fraud-risk-platform/models` |
| `FRP_EVIDENCE_ROOT` | `…/fraud-risk-platform/docs/evidence` |

In Colab each root must be absolute and under the Drive mount, otherwise it would silently be ephemeral. Only the root folders are created here; no `bronze/silver/gold` structure (that is a later layer). The variables are exported to this kernel and its subprocesses; re-run the notebook in each new session.

In [ ]:
DRIVE_BASE = Path(os.environ.get("FRP_DRIVE_BASE", f"{MYDRIVE}/fraud-risk-platform"))

DEFAULT_ROOTS = {
    "FRP_DATA_ROOT": DRIVE_BASE / "data",
    "FRP_MODELS_ROOT": DRIVE_BASE / "models",
    "FRP_EVIDENCE_ROOT": DRIVE_BASE / "docs" / "evidence",
}

ROOTS = {}
for name, default in DEFAULT_ROOTS.items():
    path = Path(os.environ.get(name, str(default))).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path, got: {path}")
    if IN_COLAB and not str(path).startswith(DRIVE_MOUNT + "/"):
        raise RuntimeError(
            f"{name}={path} is not under the Drive mount ({DRIVE_MOUNT}); it would not survive a "
            "session reset (ADR-0012)."
        )
    path.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(path)
    ROOTS[name] = path

for name, path in ROOTS.items():
    print(f"{name:<18} = {path}   [exists={path.is_dir()}, writable={os.access(path, os.W_OK)}]")

## 6 · Gate P0 smoke test (write → fresh read-back → SHA-256 → cleanup)

Required by ADR-0012 and §8.4: Drive's FUSE layer may serve a cached view right after a write, so verification must **not** reuse the write handle.

For each root (the data root first) the cell:

1. writes a unique random payload to `<root>/_smoke_test.tmp` and flushes it (`fsync` is attempted, but its absence on Drive is tolerated and not treated as a durability guarantee);
2. re-`stat`s the path and re-opens it with a **new** file handle;
3. compares the byte size and SHA-256 with the bytes that were written;
4. deletes only that test file and confirms it is gone.

The cell raises if any root fails, so a green run means every root passed.

**Limit of this check:** a read-back inside the same process can be served from the local kernel page cache. A pass proves the write was accepted and that a fresh handle reads the expected bytes; it does not by itself prove the bytes reached Google's servers. Stronger evidence (reading the data from a new session after a runtime reset) belongs to the Phase 2 Drive publication tests.

In [ ]:
import hashlib
import time
import uuid

SMOKE_NAME = "_smoke_test.tmp"


def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_of_file(path: Path) -> str:
    """Hash a file through a freshly opened handle."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def smoke_test(root: Path) -> dict:
    target = root / SMOKE_NAME
    payload = f"frp-smoke {uuid.uuid4()} {time.time_ns()}\n".encode() + os.urandom(4096)
    result = {"root": str(root), "bytes": len(payload), "expected_sha256": sha256_hex(payload)}
    try:
        with open(target, "wb") as fh:  # write handle
            fh.write(payload)
            fh.flush()
            try:
                os.fsync(fh.fileno())
                result["fsync"] = "ok"
            except OSError as exc:
                result["fsync"] = f"unsupported (errno {exc.errno}) - tolerated"
        # write handle is closed; everything below uses fresh stat/open calls
        size_on_disk = os.stat(target).st_size
        result["size_on_disk"] = size_on_disk
        result["actual_sha256"] = sha256_of_file(target)
        result["size_ok"] = size_on_disk == len(payload)
        result["hash_ok"] = result["actual_sha256"] == result["expected_sha256"]
    except Exception as exc:  # report, do not hide
        result["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        try:
            target.unlink()  # remove only our own test object
        except FileNotFoundError:
            pass
        except Exception as exc:
            result["cleanup_error"] = f"{type(exc).__name__}: {exc}"
    result["cleaned_up"] = not target.exists()
    result["passed"] = (
        "error" not in result
        and result.get("size_ok") is True
        and result.get("hash_ok") is True
        and result["cleaned_up"]
    )
    return result


results = {name: smoke_test(path) for name, path in ROOTS.items()}

print("=== Workspace (ephemeral, recreated each session) ===")
print(f"  session disk : {SESSION_DIR}")
print(f"  project root : {PROJECT_ROOT}")
print(f"  virtualenv   : {venv_prefix}")
print("\n=== Persistent roots (Drive) ===")
for name, r in results.items():
    status = "PASS" if r["passed"] else "FAIL"
    print(f"  [{status}] {name:<18} {r['root']}")
    print(f"           bytes={r['bytes']} size_on_disk={r.get('size_on_disk')} fsync={r.get('fsync')}")
    print(f"           sha256 expected={r['expected_sha256'][:16]}...  read-back={str(r.get('actual_sha256'))[:16]}...")
    if "error" in r or "cleanup_error" in r:
        print(f"           error: {r.get('error') or r.get('cleanup_error')}")
    print(f"           test file removed: {r['cleaned_up']}")

failed = [n for n, r in results.items() if not r["passed"]]
if failed:
    raise RuntimeError(f"Gate P0 smoke test FAILED for: {', '.join(failed)}")

print("\nGate P0 smoke test PASSED: write / fresh read-back / SHA-256 / cleanup verified on all roots.")
print("Reminder: the session disk and .venv are ephemeral - re-run this notebook after any runtime reset.")

## 7 · Optional: cross-check with `colab_setup.py`

`colab_setup.py` is a required project deliverable (§8.5.1). If it exists in the checkout, this cell runs it so the script and the notebook can be compared side by side. The script must be idempotent and must **detect the existing Drive mount** (it runs in a subprocess, which cannot mount Drive itself). If the script does not exist yet, the cell says so and the cells above remain the authoritative equivalent.

In [ ]:
setup_script = PROJECT_ROOT / "colab_setup.py"
if setup_script.is_file():
    print(f"Found {setup_script} - running it with the roots resolved above.")
    run([sys.executable, setup_script], cwd=PROJECT_ROOT, env=os.environ.copy())
else:
    print("colab_setup.py not found in the checkout: nothing to cross-check.")
    print("This notebook is the equivalent bootstrap; Gate P0 still requires the script as a deliverable.")